# QVI Chip Category Analysis
**Analyst:** Chloe Truong  
**Date:** July 2026  
**Objective:** Understand customer chip purchasing behaviour across segments to support the upcoming category review for Julia (Category Manager).

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

# Load raw data
transactions = pd.read_excel('~/Downloads/QVI_transaction_data.xlsx')
purchase_behaviour = pd.read_csv('~/Downloads/QVI_purchase_behaviour.csv')

print(f'Transactions: {transactions.shape[0]:,} rows × {transactions.shape[1]} columns')
print(f'Purchase Behaviour: {purchase_behaviour.shape[0]:,} rows × {purchase_behaviour.shape[1]} columns')

## 2. High-Level Data Checks

In [ ]:
print('=== TRANSACTIONS - First Look ===')
display(transactions.head())
display(transactions.dtypes)
print('\nMissing values:')
print(transactions.isnull().sum())

In [ ]:
print('=== PURCHASE BEHAVIOUR - First Look ===')
display(purchase_behaviour.head())
print('\nLifestage segments:', purchase_behaviour['LIFESTAGE'].unique())
print('Premium tiers:', purchase_behaviour['PREMIUM_CUSTOMER'].unique())
print('\nMissing values:')
print(purchase_behaviour.isnull().sum())

## 3. Data Cleaning & Format Corrections

In [ ]:
# DATE is stored as Excel serial number — convert to datetime
# Excel epoch starts 1899-12-30
transactions['DATE'] = pd.to_datetime(transactions['DATE'], unit='D', origin='1899-12-30')

print('Date range:', transactions['DATE'].min().date(), 'to', transactions['DATE'].max().date())
print('\nSample dates:')
print(transactions['DATE'].head())

In [ ]:
# Check for non-chip products in PROD_NAME
# Chips should have gram weights; flag products without numbers
non_chip_mask = ~transactions['PROD_NAME'].str.contains(r'\d', na=False)
print('Products with no gram weight (possible non-chips):')
print(transactions.loc[non_chip_mask, 'PROD_NAME'].unique())

In [ ]:
# Remove salsa products — not chips
salsa_mask = transactions['PROD_NAME'].str.contains('salsa', case=False, na=False)
print(f'Salsa rows to remove: {salsa_mask.sum():,}')
transactions = transactions[~salsa_mask].copy()
print(f'Transactions after removing salsa: {len(transactions):,}')

In [ ]:
# Check for duplicate transactions
dupes = transactions.duplicated().sum()
print(f'Duplicate rows: {dupes}')
transactions = transactions.drop_duplicates()

## 4. Outlier Detection & Removal

In [ ]:
print('=== PROD_QTY Summary ===')
print(transactions['PROD_QTY'].describe())

print('\n=== TOT_SALES Summary ===')
print(transactions['TOT_SALES'].describe())

In [ ]:
# Check extreme PROD_QTY values
high_qty = transactions[transactions['PROD_QTY'] > 20]
print(f'Transactions with PROD_QTY > 20: {len(high_qty)}')
print(high_qty[['DATE','LYLTY_CARD_NBR','PROD_NAME','PROD_QTY','TOT_SALES']])

In [ ]:
# Investigate the loyalty card buying 200 units
suspect_cards = transactions[transactions['PROD_QTY'] > 20]['LYLTY_CARD_NBR'].unique()
print('Full purchase history of suspicious customers:')
display(transactions[transactions['LYLTY_CARD_NBR'].isin(suspect_cards)]
        .sort_values(['LYLTY_CARD_NBR','DATE']))

In [ ]:
# Card buying 200 units consistently is likely a commercial buyer (not a retail customer)
# Remove these cards from analysis
before = len(transactions)
transactions = transactions[~transactions['LYLTY_CARD_NBR'].isin(suspect_cards)].copy()
print(f'Removed {before - len(transactions):,} rows from commercial buyers')
print(f'Remaining transactions: {len(transactions):,}')

In [ ]:
# Verify no more outliers
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
transactions['PROD_QTY'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Product Quantity')
axes[0].set_xlabel('Units Purchased')

transactions['TOT_SALES'].hist(bins=30, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribution of Total Sales ($)')
axes[1].set_xlabel('Sale Value ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Check transaction count by month — ensure full 12-month coverage
transactions['MONTH'] = transactions['DATE'].dt.to_period('M')
monthly = transactions.groupby('MONTH').size().reset_index(name='TXN_COUNT')

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(monthly['MONTH'].astype(str), monthly['TXN_COUNT'], color='steelblue', edgecolor='white')
ax.set_title('Monthly Transaction Volume')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Transactions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nNote: December dip is expected — shorter data window, not a true sales drop.')

## 5. Feature Engineering: Pack Size & Brand Name

In [ ]:
# Extract PACK_SIZE — the numeric gram value in product name
transactions['PACK_SIZE'] = (
    transactions['PROD_NAME']
    .str.extract(r'(\d+)(?=g)', expand=False)
    .astype(float)
)

print('Pack sizes found (grams):')
print(sorted(transactions['PACK_SIZE'].dropna().unique()))
print(f'\nMissing pack sizes: {transactions["PACK_SIZE"].isnull().sum()}')

In [ ]:
# Pack size distribution
fig, ax = plt.subplots(figsize=(13, 4))
transactions['PACK_SIZE'].value_counts().sort_index().plot(kind='bar', ax=ax, 
                                                            color='steelblue', edgecolor='white')
ax.set_title('Transactions by Pack Size (grams)')
ax.set_xlabel('Pack Size (g)')
ax.set_ylabel('Number of Transactions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Extract BRAND — first word of product name, with known cleaning
transactions['BRAND'] = transactions['PROD_NAME'].str.split().str[0].str.upper()

# Standardise brand name variants
brand_map = {
    'RED': 'RRD',          # Red Rock Deli abbreviation inconsistency
    'SNBTS': 'SUNBITES',
    'INFZNS': 'INFUZIONS',
    'DORITO': 'DORITOS',
    'GRAIN': 'GRNWVES',    # Grain Waves
    'SMITH': 'SMITHS',
    'NCC': 'NATURAL',
    'WW': 'WOOLWORTHS',
}
transactions['BRAND'] = transactions['BRAND'].replace(brand_map)

print('Unique brands after cleaning:')
print(sorted(transactions['BRAND'].unique()))

In [ ]:
# Top brands by transaction volume
brand_sales = (transactions.groupby('BRAND')
               .agg(TRANSACTIONS=('TXN_ID','count'), REVENUE=('TOT_SALES','sum'))
               .sort_values('REVENUE', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
brand_sales['TRANSACTIONS'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Transactions by Brand')
axes[0].set_xlabel('Number of Transactions')

brand_sales['REVENUE'].sort_values().plot(kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Revenue by Brand ($)')
axes[1].set_xlabel('Total Revenue ($)')

plt.tight_layout()
plt.show()

## 6. Merge Datasets & Define Customer Segment Metrics

In [ ]:
# Merge on loyalty card number
df = transactions.merge(purchase_behaviour, on='LYLTY_CARD_NBR', how='left')

print(f'Merged dataset shape: {df.shape}')
print(f'Rows missing segment info: {df["LIFESTAGE"].isnull().sum():,}')

# Drop rows with no segment match
df = df.dropna(subset=['LIFESTAGE', 'PREMIUM_CUSTOMER'])
print(f'Final analysis dataset: {len(df):,} rows')

In [ ]:
# Define segment-level metrics
segment_metrics = (
    df.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])
    .agg(
        TOTAL_SALES        = ('TOT_SALES', 'sum'),
        TOTAL_CUSTOMERS    = ('LYLTY_CARD_NBR', 'nunique'),
        TOTAL_TRANSACTIONS = ('TXN_ID', 'count'),
        TOTAL_UNITS        = ('PROD_QTY', 'sum'),
    )
    .reset_index()
)

# Derived metrics
segment_metrics['AVG_SPEND_PER_TXN']      = (segment_metrics['TOTAL_SALES'] / segment_metrics['TOTAL_TRANSACTIONS']).round(2)
segment_metrics['AVG_UNITS_PER_TXN']      = (segment_metrics['TOTAL_UNITS'] / segment_metrics['TOTAL_TRANSACTIONS']).round(2)
segment_metrics['AVG_TXN_PER_CUSTOMER']   = (segment_metrics['TOTAL_TRANSACTIONS'] / segment_metrics['TOTAL_CUSTOMERS']).round(2)
segment_metrics['AVG_SPEND_PER_CUSTOMER'] = (segment_metrics['TOTAL_SALES'] / segment_metrics['TOTAL_CUSTOMERS']).round(2)
segment_metrics['AVG_PRICE_PER_UNIT']     = (segment_metrics['TOTAL_SALES'] / segment_metrics['TOTAL_UNITS']).round(2)

display(segment_metrics.sort_values('TOTAL_SALES', ascending=False))

## 7. Segment Analysis: Who Drives Chip Sales?

In [ ]:
# Total sales by segment — heatmap
sales_pivot = segment_metrics.pivot(index='LIFESTAGE', columns='PREMIUM_CUSTOMER', values='TOTAL_SALES')
sales_pivot = sales_pivot[['Budget', 'Mainstream', 'Premium']]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(sales_pivot, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10})
ax.set_title('Total Chip Sales ($) by Customer Segment', fontsize=13, pad=15)
ax.set_xlabel('Customer Tier')
ax.set_ylabel('Lifestage')
plt.tight_layout()
plt.show()

In [ ]:
# Number of customers per segment
cust_pivot = segment_metrics.pivot(index='LIFESTAGE', columns='PREMIUM_CUSTOMER', values='TOTAL_CUSTOMERS')
cust_pivot = cust_pivot[['Budget', 'Mainstream', 'Premium']]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(cust_pivot, annot=True, fmt=',.0f', cmap='Blues', ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10})
ax.set_title('Number of Customers by Segment', fontsize=13, pad=15)
ax.set_xlabel('Customer Tier')
ax.set_ylabel('Lifestage')
plt.tight_layout()
plt.show()

In [ ]:
# Average spend per customer — who is most valuable per head?
spend_pivot = segment_metrics.pivot(index='LIFESTAGE', columns='PREMIUM_CUSTOMER', values='AVG_SPEND_PER_CUSTOMER')
spend_pivot = spend_pivot[['Budget', 'Mainstream', 'Premium']]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(spend_pivot, annot=True, fmt='.2f', cmap='Greens', ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10})
ax.set_title('Avg Annual Spend per Customer ($) by Segment', fontsize=13, pad=15)
ax.set_xlabel('Customer Tier')
ax.set_ylabel('Lifestage')
plt.tight_layout()
plt.show()

In [ ]:
# Average price per unit — willingness to pay by segment
price_pivot = segment_metrics.pivot(index='LIFESTAGE', columns='PREMIUM_CUSTOMER', values='AVG_PRICE_PER_UNIT')
price_pivot = price_pivot[['Budget', 'Mainstream', 'Premium']]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(price_pivot, annot=True, fmt='.2f', cmap='Purples', ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10})
ax.set_title('Avg Price per Unit ($) by Segment — Willingness to Pay', fontsize=13, pad=15)
ax.set_xlabel('Customer Tier')
ax.set_ylabel('Lifestage')
plt.tight_layout()
plt.show()

## 8. Deep Dive: Top Segments

In [ ]:
# Focus on top 3 revenue segments
top3 = segment_metrics.nlargest(3, 'TOTAL_SALES')[['LIFESTAGE','PREMIUM_CUSTOMER','TOTAL_SALES',
                                                     'TOTAL_CUSTOMERS','AVG_SPEND_PER_CUSTOMER',
                                                     'AVG_TXN_PER_CUSTOMER','AVG_PRICE_PER_UNIT']]
print('=== TOP 3 REVENUE-GENERATING SEGMENTS ===')
display(top3)

In [ ]:
# Brand preference by top segments
top_segs = top3[['LIFESTAGE','PREMIUM_CUSTOMER']].values.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (ls, pc) in zip(axes, top_segs):
    seg_df = df[(df['LIFESTAGE'] == ls) & (df['PREMIUM_CUSTOMER'] == pc)]
    brand_rev = seg_df.groupby('BRAND')['TOT_SALES'].sum().nlargest(8).sort_values()
    brand_rev.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{ls}\n{pc}', fontsize=9)
    ax.set_xlabel('Revenue ($)')
    ax.set_ylabel('')

plt.suptitle('Top Brands in Top 3 Segments', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pack size preference by top segments
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (ls, pc) in zip(axes, top_segs):
    seg_df = df[(df['LIFESTAGE'] == ls) & (df['PREMIUM_CUSTOMER'] == pc)]
    pack_rev = seg_df.groupby('PACK_SIZE')['TOT_SALES'].sum().sort_index()
    pack_rev.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_title(f'{ls}\n{pc}', fontsize=9)
    ax.set_xlabel('Pack Size (g)')
    ax.set_ylabel('Revenue ($)')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Pack Size Preference in Top 3 Segments', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Mainstream Young Singles/Couples — do they pay more per unit than other mainstream segments?
mainstream = df[df['PREMIUM_CUSTOMER'] == 'Mainstream']
price_by_life = mainstream.groupby('LIFESTAGE')['TOT_SALES'].sum() / mainstream.groupby('LIFESTAGE')['PROD_QTY'].sum()
price_by_life = price_by_life.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 4))
price_by_life.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Avg Price per Unit — Mainstream Customers by Lifestage')
ax.set_xlabel('Lifestage')
ax.set_ylabel('Avg Price per Unit ($)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print(price_by_life.round(2))

## 9. Key Insights & Strategic Recommendation

### Summary of Findings

**1. Biggest revenue segments by absolute sales:**  
- **Mainstream Young Singles/Couples** — large customer base, high transaction frequency  
- **Mainstream Retirees** — consistent buyers, moderate basket size  
- **Mainstream Older Singles/Couples**  

**2. Highest spend per customer:**  
Older Families and Young Families in Budget/Mainstream tiers buy more units per trip, likely buying for households.  

**3. Price sensitivity:**  
- Premium customers pay more per unit — room to range premium SKUs for these segments  
- Mainstream Young Singles/Couples pay *above-average* price per unit vs other Mainstream segments — they are willing to pay for quality chips despite not being labelled "Premium"  

**4. Pack size:**  
- 175g is the dominant pack size across all segments  
- Families skew toward larger packs (330g–380g) — opportunity to grow these SKUs in Family aisles  

**5. Brand preference:**  
- Kettle and Smiths dominate across most segments  
- Mainstream Young Singles/Couples show stronger affinity for Kettle (premium taste positioning at mid-price)  

---

### Recommendation to Julia

> **Target Mainstream Young Singles/Couples as the primary growth segment.**  
> They represent the largest customer pool, transact frequently, and are willing to pay above-average prices for chips — suggesting they respond to product quality over pure price. Prioritise shelf space and promotions for Kettle-style premium-but-accessible brands in 150g–175g packs.  
>  
> **For families (Budget & Mainstream), drive volume through value multi-packs.**  
> Their high units-per-trip behaviour makes them ideal targets for bulk-pack promotions and aisle placement near household staples.  
>  
> **Retirees are a stable, loyal base — protect with consistent ranging of familiar brands (Smiths, Doritos).**